# Sesión 5 · Explorar y pronosticar: la economía en 200 trimestres
### Programación y Analítica de Datos · Programa de Economía

> «La predicción es muy difícil, sobre todo si se trata del futuro.» — dicho atribuido al físico Niels Bohr, y a varios otros

Hoy no hay que memorizar nada. En este notebook tiene **cincuenta años de la economía de Estados Unidos**, trimestre por trimestre: producción, consumo, inversión, precios, desempleo, tasas de interés. La tarea es **explorarlos y jugar**: mover deslizadores, elegir variables, poner a competir modelos de *machine learning* y ver cuál pronostica mejor.

Al final de la clase usted habrá hecho, sin ser experto, lo mismo que hace un equipo de análisis en un banco central: mirar los datos, buscar relaciones, entrenar modelos, medir su error y desconfiar de ellos con argumentos.

| Símbolo | Qué significa |
|:-:|:--|
| ▶️ | Celda de código: se ejecuta con el botón de la izquierda o con **Shift + Enter** |
| 🔮 | **Prediga** antes de ejecutar: anote en su cuaderno qué cree que va a pasar |
| ✍️ | **Su turno**: reemplace los tres puntos `...` por su respuesta |
| ✅ | **Comprobación**: le dice si su turno quedó bien, y le da una pista si no |
| 🎚️ | **Juegue**: mueva los deslizadores y las listas, y mire qué cambia |
| 🧭 | **Reto**: una pregunta abierta para explorar por su cuenta |

| Parte | Qué hay |
|:--|:--|
| 1 · Conocer la base | Qué hay en la tabla y qué significa cada columna |
| 2 · Recorrer el tiempo | Cualquier variable, cualquier periodo, con las recesiones marcadas |
| 3 · Buscar relaciones | La curva de Phillips, la ley de Okun y todas las correlaciones |
| 4 · La arena de modelos | Cinco modelos de *machine learning* pronostican el próximo trimestre |
| 5 · Cuando el modelo memoriza | El sobreajuste, en vivo |
| 6 · ¿Qué mira el modelo? | Las variables que más pesan en el pronóstico |
| 7 · El torneo | Todos los modelos contra todas las variables |
| 8 · Simulador | Invente una economía y pida un pronóstico |
| 9 · Lo que un modelo no ve | Límites y responsabilidad |

**Qué hace la celda siguiente:** trae las herramientas, carga la base de datos y la traduce al español. Las librerías ya vienen instaladas en Colab:

- **statsmodels**, la librería de econometría de Python. Además de modelos, trae bases de datos de economía listas para usar; de ahí sale la de hoy.
- **scikit-learn**, la librería de *machine learning* más usada del mundo. De ella salen los cinco modelos de la parte 4.

Basta con **ejecutarla una vez** y esperar el mensaje con ✅.

In [ ]:
#@title ⚙️ Preparación · ejecute esta celda primero (no necesita leerla)

# ---------------------------------------------------------------------------
# 1. HERRAMIENTAS: pandas maneja tablas, matplotlib dibuja gráficos,
#    ipywidgets pone los deslizadores con los que usted moverá las variables.
# ---------------------------------------------------------------------------
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

VERDE, CORAL, GRIS, AZUL = "#2A9D8F", "#E76F51", "#B8C1CC", "#1F3A5F"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#E6EAEE", "axes.axisbelow": True, "font.size": 11})

# ---------------------------------------------------------------------------
# 2. FORMATO: escribir pesos y porcentajes como se escriben en Colombia.
# ---------------------------------------------------------------------------
def pesos(x):
    """Convierte 1423500 en el texto $1.423.500"""
    return "$" + f"{x:,.0f}".replace(",", ".")

def porcentaje(x, decimales=2):
    """Convierte 0.0419 en el texto 4,19 %"""
    return f"{x * 100:.{decimales}f}".replace(".", ",") + " %"

def numero(x, decimales=0):
    """Convierte 1234.5 en el texto 1.234,5"""
    return f"{x:,.{decimales}f}".replace(",", "X").replace(".", ",").replace("X", ".")

# ---------------------------------------------------------------------------
# 3. DESLIZADORES: controles para "jugar" con las variables sin escribir código.
# ---------------------------------------------------------------------------
try:
    import ipywidgets as widgets
    HAY_DESLIZADORES = os.environ.get("NB_SIN_DESLIZADORES") != "1"
except ImportError:
    HAY_DESLIZADORES = False

_ESTILO = {"description_width": "initial"}

def deslizador(nombre, minimo, maximo, paso, valor):
    """Crea una barra deslizante: nombre visible, mínimo, máximo, tamaño del paso y valor inicial."""
    if not HAY_DESLIZADORES:
        return valor
    es_entero = all(float(v).is_integer() for v in (minimo, maximo, paso, valor))
    clase = widgets.IntSlider if es_entero else widgets.FloatSlider
    return clase(description=nombre, min=minimo, max=maximo, step=paso, value=valor, continuous_update=False,
                 style=_ESTILO, layout=widgets.Layout(width="560px"))

def lista(nombre, opciones, valor=None):
    """Crea una lista desplegable para elegir una opción."""
    valor = opciones[0] if valor is None else valor
    if not HAY_DESLIZADORES:
        return valor
    return widgets.Dropdown(description=nombre, options=opciones, value=valor, style=_ESTILO,
                            layout=widgets.Layout(width="420px"))

def jugar(funcion, **controles):
    """Conecta una función con sus controles: cada vez que usted mueve uno, la función se vuelve a ejecutar."""
    if HAY_DESLIZADORES:
        widgets.interact(funcion, **controles)
    else:
        funcion(**controles)
        print("\n(En este entorno no hay deslizadores: cambie los valores iniciales en la celda y ejecútela de nuevo.)")

import warnings
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
warnings.filterwarnings("ignore")

# --- La base: 203 trimestres de la economía de Estados Unidos, 1959-2009 ---
crudo = sm.datasets.macrodata.load_pandas().data
fechas = pd.period_range("1959Q1", periods=len(crudo), freq="Q").to_timestamp()

eco = pd.DataFrame(index=fechas)
eco["pib"] = crudo["realgdp"].values                  # miles de millones de dólares de 2005, ritmo anual
eco["consumo"] = crudo["realcons"].values
eco["inversion"] = crudo["realinv"].values
eco["gasto_gobierno"] = crudo["realgovt"].values
eco["ingreso_disponible"] = crudo["realdpi"].values
eco["ipc"] = crudo["cpi"].values                      # índice de precios, 1982-84 = 100
eco["dinero_m1"] = crudo["m1"].values                 # miles de millones de dólares corrientes
eco["tasa_interes"] = crudo["tbilrate"].values        # % anual, bonos del Tesoro a 3 meses
eco["desempleo"] = crudo["unemp"].values              # % de la fuerza laboral
eco["poblacion"] = crudo["pop"].values                # millones de personas
eco["inflacion"] = crudo["infl"].values               # % anualizado, trimestre contra trimestre
eco["tasa_real"] = crudo["realint"].values            # tasa de interés menos inflación

# Recesiones de Estados Unidos según el NBER (trimestre del pico y trimestre del valle)
RECESIONES = [("1960-04-01", "1961-01-01"), ("1969-10-01", "1970-10-01"), ("1973-10-01", "1975-01-01"),
              ("1980-01-01", "1980-07-01"), ("1981-07-01", "1982-10-01"), ("1990-07-01", "1991-01-01"),
              ("2001-01-01", "2001-10-01"), ("2007-10-01", "2009-04-01")]

def sombrear_recesiones(ax, desde=None, hasta=None):
    """Pinta de gris los trimestres de recesión."""
    for ini, fin in RECESIONES:
        ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fin) + pd.offsets.QuarterEnd(0), color="#B8C1CC", alpha=0.35, lw=0)

UNIDADES = {"pib": "miles de millones de US$ de 2005", "consumo": "miles de millones de US$ de 2005",
            "inversion": "miles de millones de US$ de 2005", "gasto_gobierno": "miles de millones de US$ de 2005",
            "ingreso_disponible": "miles de millones de US$ de 2005", "ipc": "índice (1982-84 = 100)",
            "dinero_m1": "miles de millones de US$", "tasa_interes": "% anual", "desempleo": "% de la fuerza laboral",
            "poblacion": "millones de personas", "inflacion": "% anualizado", "tasa_real": "% anual",
            "crec_pib": "% frente a hace un año", "crec_consumo": "% frente a hace un año",
            "crec_inversion": "% frente a hace un año", "crec_ingreso": "% frente a hace un año",
            "crec_dinero": "% frente a hace un año", "cambio_desempleo": "puntos frente al trimestre anterior"}

def comprobar(condicion, bien, pista):
    """Revisa un turno: si la condición se cumple, felicita; si no, da una pista."""
    print("✅ " + bien if condicion else "🔁 Todavía no. Pista: " + pista)

def dec(x, d=2):
    """Escribe un número con coma decimal."""
    return numero(x, d)

print("✅ Herramientas listas y base cargada:", len(eco), "trimestres,", eco.shape[1], "variables.")

---
## 1 · Conocer la base

La base se llama `macrodata` y viene incluida en `statsmodels`. La armó Skipper Seabold, uno de los creadores de la librería, con datos oficiales: la Reserva Federal de San Luis (FRED) y la Oficina de Estadísticas Laborales de Estados Unidos (BLS). Va del primer trimestre de 1959 al tercero de 2009. La usamos porque es pública, está validada y cubre ocho recesiones, incluida la crisis financiera de 2008.

| Columna | Qué mide, en palabras sencillas |
|:--|:--|
| `pib` | **Producto interno bruto real**: todo lo que el país produjo en el trimestre, sin el efecto de la inflación |
| `consumo` | Lo que gastan los hogares: comida, ropa, transporte, servicios |
| `inversion` | Lo que gastan las empresas en máquinas, edificios y vivienda nueva. Es la parte más nerviosa del PIB |
| `gasto_gobierno` | Compras e inversión del gobierno federal |
| `ingreso_disponible` | Lo que les queda a los hogares después de impuestos |
| `ipc` | **Índice de precios al consumidor**: cuánto cuesta una canasta fija de compras |
| `dinero_m1` | El dinero en efectivo y en cuentas corrientes que circula |
| `tasa_interes` | Lo que paga el gobierno por prestar a 3 meses. La mueve la Reserva Federal para frenar o acelerar la economía |
| `desempleo` | Porcentaje de las personas que buscan trabajo y no lo consiguen |
| `poblacion` | Habitantes, en millones |
| `inflacion` | Qué tan rápido suben los precios (el `ipc`), en % anualizado |
| `tasa_real` | Tasa de interés menos inflación: lo que de verdad gana quien presta |

> Las cifras en dólares son **reales**, a precios de 2005: se les quitó la inflación para que 1960 y 2009 se puedan comparar.

In [ ]:
eco.head()

`describe()` resume cada columna: cuántos datos tiene (`count`), el promedio (`mean`), la desviación estándar (`std`), el mínimo, los cuartiles y el máximo. Recuerde que pandas escribe los decimales con **punto**.

In [ ]:
eco.describe().round(2).T

🔮 **Prediga**: ¿en qué trimestre cree que el desempleo fue más alto en estos cincuenta años? ¿Y la inflación? Anótelo y ejecute.

In [ ]:
for columna in ["desempleo", "inflacion", "tasa_interes"]:
    fecha = eco[columna].idxmax()                    # la fecha del valor máximo
    print(f"{columna:>13}: máximo de {dec(eco[columna].max())} % en {fecha.year}, trimestre {fecha.quarter}")

---
## 2 · Recorrer el tiempo

🎚️ **Juegue.** Elija una variable y un periodo. Las franjas grises son las **recesiones** según el NBER, la institución que en Estados Unidos dice oficialmente cuándo empieza y cuándo termina una.

Active **«crecimiento anual»** para ver cuánto creció cada trimestre frente al mismo trimestre del año anterior: en las variables que solo suben (PIB, precios, población), el crecimiento es lo que de verdad cuenta la historia.

In [ ]:
def recorrer(variable="desempleo", desde=1959, hasta=2009, ver="nivel"):
    serie = eco[variable]
    unidad = UNIDADES[variable]
    if ver == "crecimiento anual":
        serie = serie.pct_change(4) * 100                # contra 4 trimestres atrás
        unidad = "% frente a hace un año"
    serie = serie[str(desde):str(hasta)]
    fig, ax = plt.subplots(figsize=(11, 4))
    sombrear_recesiones(ax)
    ax.plot(serie.index, serie, color=AZUL, linewidth=2)
    ax.set_xlim(serie.index[0], serie.index[-1])
    ax.set_title(f"{variable} ({ver}), {desde}-{hasta}  ·  gris = recesión")
    ax.set_ylabel(unidad)
    plt.show()
    print(f"Promedio del periodo: {dec(serie.mean())}  ·  mínimo: {dec(serie.min())}  ·  máximo: {dec(serie.max())}")

jugar(recorrer,
      variable=lista("Variable", list(eco.columns), "desempleo"),
      desde=deslizador("Desde", 1959, 2008, 1, 1959),
      hasta=deslizador("Hasta", 1960, 2009, 1, 2009),
      ver=lista("Ver", ["nivel", "crecimiento anual"]))

🧭 **Reto.** Con `desempleo`, mire qué pasa **durante** y **después** de cada franja gris. ¿El desempleo sube antes, durante o después de la recesión? Luego mire `inversion` con «crecimiento anual»: ¿por qué cree que la inversión cae mucho más que el consumo en las recesiones?

### Las variables que usarán los modelos

Para pronosticar, los modelos no usan los niveles (el PIB de 2009 es cinco veces el de 1959, y eso no dice nada del próximo trimestre). Usan **cambios**: cuánto creció cada variable en el último año y cuánto cambió el desempleo frente al trimestre anterior. La celda siguiente los crea.

In [ ]:
eco["crec_pib"] = eco["pib"].pct_change(4) * 100
eco["crec_consumo"] = eco["consumo"].pct_change(4) * 100
eco["crec_inversion"] = eco["inversion"].pct_change(4) * 100
eco["crec_ingreso"] = eco["ingreso_disponible"].pct_change(4) * 100
eco["crec_dinero"] = eco["dinero_m1"].pct_change(4) * 100
eco["cambio_desempleo"] = eco["desempleo"].diff()          # diferencia con el trimestre anterior

eco[["crec_pib", "crec_inversion", "cambio_desempleo"]].tail(6).round(2)

Fíjese en 2008 y 2009: el PIB cae frente al año anterior (números negativos), la inversión se desploma más de 20 % y el desempleo sube casi un punto por trimestre. Es la crisis financiera, vista en tres columnas.

---
## 3 · Buscar relaciones

Dos de las relaciones más famosas de la macroeconomía se pueden ver con esta base:

- **La curva de Phillips** (A. W. Phillips, 1958): cuando el desempleo es bajo, la inflación tiende a subir, porque las empresas compiten por trabajadores y suben salarios y precios.
- **La ley de Okun** (Arthur Okun, 1962): cuando la economía crece rápido, el desempleo baja; cuando crece poco o cae, sube.

🎚️ **Juegue.** Elija qué poner en cada eje. El color indica la década. Empiece con las dos relaciones de arriba y luego invente las suyas.

In [ ]:
def relacion(eje_x="desempleo", eje_y="inflacion", decada="todas"):
    datos = eco[[eje_x, eje_y]].dropna()
    datos["decada"] = (datos.index.year // 10) * 10
    if decada != "todas":
        datos = datos[datos["decada"] == int(decada)]
    fig, ax = plt.subplots(figsize=(8, 5.5))
    puntos = ax.scatter(datos[eje_x], datos[eje_y], c=datos["decada"], cmap="viridis", s=28, alpha=0.85)
    pendiente, intercepto = np.polyfit(datos[eje_x], datos[eje_y], 1)      # la recta que mejor se ajusta
    xs = np.linspace(datos[eje_x].min(), datos[eje_x].max(), 50)
    ax.plot(xs, intercepto + pendiente * xs, color=CORAL, linewidth=2.5)
    fig.colorbar(puntos, label="década")
    r = datos[eje_x].corr(datos[eje_y])
    ax.set_title(f"{eje_y} frente a {eje_x}  ·  correlación = {dec(r)}")
    ax.set_xlabel(f"{eje_x} ({UNIDADES[eje_x]})")
    ax.set_ylabel(f"{eje_y} ({UNIDADES[eje_y]})")
    plt.show()
    fuerza = "fuerte" if abs(r) > 0.6 else "moderada" if abs(r) > 0.3 else "débil"
    sentido = "van en la misma dirección" if r > 0 else "van en direcciones opuestas"
    print(f"Correlación {fuerza}: cuando {eje_x} sube, {eje_y} tiende a {'subir' if r > 0 else 'bajar'} ({sentido}).")
    print(f"Pendiente de la recta: {dec(pendiente)} puntos de {eje_y} por cada punto de {eje_x}.")

columnas = list(eco.columns)
jugar(relacion,
      eje_x=lista("Eje horizontal", columnas, "desempleo"),
      eje_y=lista("Eje vertical", columnas, "inflacion"),
      decada=lista("Década", ["todas", "1960", "1970", "1980", "1990", "2000"]))

**Para leer lo que salió.** La **correlación** va de −1 a 1. Cerca de 1: las dos variables suben juntas. Cerca de −1: cuando una sube, la otra baja. Cerca de 0: no hay relación en línea recta.

Con todas las décadas juntas, la curva de Phillips casi desaparece. Elija solo la **década de 1960**: ahí sí se ve. En los setenta llegó la **estanflación** (desempleo alto *y* inflación alta a la vez, por la crisis del petróleo) y la relación se rompió. Una lección para todo el día: **una relación que se cumplió en el pasado puede dejar de cumplirse**.

✍️ **Su turno.** Ahora pruebe la ley de Okun con código. `.corr()` calcula la correlación entre dos columnas. Reemplace los `...` por el nombre de la columna del **cambio del desempleo**.

In [ ]:
r_okun = eco["crec_pib"].corr(eco[...])

print("Correlación entre el crecimiento del PIB y el cambio del desempleo:", dec(r_okun))

In [ ]:
#@title ✅ Comprobación
comprobar(abs(r_okun - eco["crec_pib"].corr(eco["cambio_desempleo"])) < 1e-9,
          f"La correlación es {dec(r_okun)}: negativa y fuerte. Cuando la economía crece, el desempleo baja. Okun tenía razón.",
          'escriba el nombre entre comillas: eco["cambio_desempleo"]')

### Todas las relaciones a la vez

Un **mapa de correlaciones** pone cada par de variables en un cuadro: rojo cuando van juntas, azul cuando van en direcciones opuestas. Así se buscan pistas para los modelos.

In [ ]:
cambios = ["crec_pib", "crec_consumo", "crec_inversion", "crec_ingreso", "crec_dinero",
           "inflacion", "tasa_interes", "desempleo", "cambio_desempleo"]
matriz = eco[cambios].corr()

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(matriz, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cambios)), cambios, rotation=45, ha="right")
ax.set_yticks(range(len(cambios)), cambios)
for i in range(len(cambios)):
    for j in range(len(cambios)):
        valor = matriz.iloc[i, j]
        ax.text(j, i, f"{valor:.1f}".replace(".", ","), ha="center", va="center", fontsize=8,
                color="white" if abs(valor) > 0.6 else "black")
ax.grid(False)
fig.colorbar(im, label="correlación")
ax.set_title("¿Qué se mueve con qué?")
plt.show()

🧭 **Reto.** Busque en el mapa la pareja con la correlación **más negativa** y la más positiva (sin contar la diagonal, donde cada variable se compara consigo misma). ¿Tienen sentido económico?

---
## 4 · La arena de modelos

Ahora la pregunta de un analista: **con lo que sé hoy, ¿cuánto será el desempleo (o la inflación, o el crecimiento) el próximo trimestre?**

Así se arma el problema de *machine learning*:

- **las pistas** (en inglés, *features*): lo que se sabe hoy. Crecimiento del PIB, del consumo, de la inversión, del ingreso y del dinero; inflación, tasa de interés, desempleo y su cambio;
- **el objetivo** (*target*): el valor de la variable elegida **un trimestre después**;
- **entrenamiento y prueba**: el modelo aprende con los años anteriores al **año de corte** y se le pide pronosticar los años siguientes, que nunca vio.

Y un rival sin modelo, el **ingenuo**: «el próximo trimestre será igual a este». Un modelo que no le gane al ingenuo no sirve.

Los cinco modelos, en una frase cada uno:

| Modelo | La idea |
|:--|:--|
| **Regresión lineal** | Una suma de pistas, cada una con su peso. El clásico de la econometría |
| **Árbol de decisión** | Preguntas encadenadas: «¿el desempleo está por encima de 6? ¿la inversión cae?» |
| **Bosque aleatorio** | Trescientos árboles distintos que votan; el pronóstico es el promedio |
| **Boosting** | Árboles en fila: cada uno corrige los errores del anterior |
| **Vecinos cercanos** | Busca los cinco trimestres del pasado más parecidos a hoy y promedia lo que pasó después |

In [ ]:
PISTAS = ["desempleo", "cambio_desempleo", "inflacion", "tasa_interes",
          "crec_pib", "crec_consumo", "crec_inversion", "crec_ingreso", "crec_dinero"]

OBJETIVOS = {"Desempleo": "desempleo", "Inflación": "inflacion", "Crecimiento del PIB": "crec_pib"}

def crear_modelo(nombre):
    """Devuelve un modelo nuevo, sin entrenar."""
    if nombre == "Regresión lineal":
        return LinearRegression()
    if nombre == "Árbol de decisión":
        return DecisionTreeRegressor(max_depth=4, random_state=0)
    if nombre == "Bosque aleatorio":
        return RandomForestRegressor(n_estimators=300, random_state=0)
    if nombre == "Boosting":
        return GradientBoostingRegressor(random_state=0)
    return make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5))   # vecinos cercanos

MODELOS = ["Regresión lineal", "Árbol de decisión", "Bosque aleatorio", "Boosting", "Vecinos cercanos"]

def preparar(objetivo, corte):
    """Arma la tabla: pistas de hoy y, al lado, el objetivo del próximo trimestre. La parte en dos."""
    datos = eco[PISTAS].copy()
    datos["hoy"] = eco[objetivo]
    datos["siguiente"] = eco[objetivo].shift(-1)          # el valor del próximo trimestre
    datos = datos.dropna()
    entrena = datos[datos.index.year < corte]
    prueba = datos[datos.index.year >= corte]
    return entrena, prueba

entrena, prueba = preparar("desempleo", 2000)
print("Entrenamiento:", len(entrena), "trimestres ·", entrena.index[0].year, "a", entrena.index[-1].year)
print("Prueba:       ", len(prueba), "trimestres ·", prueba.index[0].year, "a", prueba.index[-1].year)
entrena[PISTAS + ["siguiente"]].tail(3).round(2)

🔮 **Prediga** antes de jugar: ¿cuál de las tres variables cree que será **más fácil** de pronosticar, el desempleo, la inflación o el crecimiento del PIB? ¿Y cuál modelo ganará?

🎚️ **Juegue.** Elija modelo, variable y año de corte. Mire el gráfico (línea azul: lo que pasó; coral: el pronóstico; gris punteada: el ingenuo) y la tabla de errores.

In [ ]:
def arena(modelo="Boosting", variable="Desempleo", corte=2000):
    global resultado
    objetivo = OBJETIVOS[variable]
    entrena, prueba = preparar(objetivo, corte)
    m = crear_modelo(modelo)
    m.fit(entrena[PISTAS], entrena["siguiente"])           # aprender con el pasado
    pronostico = m.predict(prueba[PISTAS])                 # pronosticar lo que no vio
    resultado = pd.DataFrame({"real": prueba["siguiente"], "pronostico": pronostico, "ingenuo": prueba["hoy"]})

    fig, ax = plt.subplots(figsize=(11, 4.2))
    sombrear_recesiones(ax)
    fechas = resultado.index + pd.offsets.QuarterBegin(1, startingMonth=1)   # el trimestre pronosticado
    ax.plot(fechas, resultado["real"], color=AZUL, linewidth=2.5, label="lo que pasó")
    ax.plot(fechas, resultado["pronostico"], color=CORAL, linewidth=2, label=f"pronóstico · {modelo}")
    ax.plot(fechas, resultado["ingenuo"], color=GRIS, linestyle="--", linewidth=1.5, label="ingenuo (igual que hoy)")
    ax.set_xlim(fechas[0], fechas[-1])
    ax.set_title(f"{variable} del próximo trimestre · aprendió con {entrena.index[0].year}-{corte - 1}")
    ax.set_ylabel(UNIDADES[objetivo])
    ax.legend()
    plt.show()

    error_m = mean_absolute_error(resultado["real"], resultado["pronostico"])
    error_i = mean_absolute_error(resultado["real"], resultado["ingenuo"])
    tabla = pd.DataFrame({"error medio (MAE)": [error_m, error_i],
                          "R²": [r2_score(resultado["real"], resultado["pronostico"]), r2_score(resultado["real"], resultado["ingenuo"])]},
                         index=[modelo, "Ingenuo"]).round(3)
    display(tabla)
    veredicto = "le GANA al ingenuo 🏆" if error_m < error_i else "PIERDE contra el ingenuo 😬"
    print(f"En promedio, {modelo} se equivoca {dec(error_m)} puntos; el ingenuo, {dec(error_i)}. El modelo {veredicto}.")

jugar(arena,
      modelo=lista("Modelo", MODELOS, "Boosting"),
      variable=lista("Pronosticar", list(OBJETIVOS), "Desempleo"),
      corte=deslizador("Año de corte", 1985, 2005, 1, 2000))

**Para leer la tabla.**

- **Error medio (MAE)**: en promedio, por cuántos puntos se equivocó el pronóstico. Si el desempleo real fue 5,0 % y el modelo dijo 5,3 %, el error de ese trimestre es 0,3 puntos. **Menor es mejor.**
- **R²**: qué parte de los movimientos explica el pronóstico. 1 es perfecto; 0, nada; **un R² negativo** quiere decir que el pronóstico es peor que decir siempre el promedio.

✍️ **Su turno.** El error medio se puede calcular con pandas, sin la librería: la diferencia entre lo real y lo pronosticado, en valor absoluto (`.abs()`, sin signo), y luego el promedio. Complete con el método que saca el **promedio**.

In [ ]:
error_medio = (resultado["real"] - resultado["pronostico"]).abs()....

print("Error medio calculado a mano:", dec(error_medio, 3), "puntos")

In [ ]:
#@title ✅ Comprobación
comprobar(abs(error_medio - mean_absolute_error(resultado["real"], resultado["pronostico"])) < 1e-9,
          "Coincide con la tabla: el MAE no es más que el promedio de los errores sin signo.",
          "el promedio en pandas es .mean()")

🧭 **Retos para la arena** (anote lo que encuentre):

1. Con el **desempleo**, ¿cuántos modelos le ganan al ingenuo? ¿Y con la **inflación**? ¿Y con el **crecimiento del PIB**?
2. Ponga el corte en **2005**: la prueba queda casi toda en la crisis de 2008. ¿Algún modelo vio venir la subida del desempleo?
3. Mueva el corte de 1985 a 2005 con **vecinos cercanos**. ¿Por qué cree que cambia tanto?

---
## 5 · Cuando el modelo memoriza

Un árbol de decisión puede hacer muchas preguntas encadenadas. La **profundidad** es cuántas preguntas seguidas puede hacer. Con pocas, el modelo es simple y quizá se queda corto. Con muchas, aprende de memoria cada trimestre del pasado, incluido el ruido, y falla con los datos nuevos. Eso se llama **sobreajuste** (*overfitting*): como el estudiante que memoriza las respuestas del parcial viejo y no entiende el tema.

🎚️ **Juegue.** Mueva la profundidad y mire las dos curvas: el error con los datos que el árbol **vio** (entrenamiento) y con los que **no vio** (prueba).

In [ ]:
entrena_d, prueba_d = preparar("desempleo", 2000)
profundidades = range(1, 16)
err_entrena, err_prueba = [], []
for p in profundidades:
    arbol = DecisionTreeRegressor(max_depth=p, random_state=0).fit(entrena_d[PISTAS], entrena_d["siguiente"])
    err_entrena.append(mean_absolute_error(entrena_d["siguiente"], arbol.predict(entrena_d[PISTAS])))
    err_prueba.append(mean_absolute_error(prueba_d["siguiente"], arbol.predict(prueba_d[PISTAS])))

def sobreajuste(profundidad=3):
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(profundidades, err_entrena, "o-", color=VERDE, label="error con lo que vio (entrenamiento)")
    ax.plot(profundidades, err_prueba, "o-", color=CORAL, label="error con lo que no vio (prueba)")
    ax.axvline(profundidad, color=AZUL, linestyle=":")
    ax.set_xlabel("profundidad del árbol (preguntas seguidas)")
    ax.set_ylabel("error medio, puntos de desempleo")
    ax.set_title("Más complejo no es mejor: el error de prueba deja de bajar")
    ax.legend()
    plt.show()
    i = profundidad - 1
    print(f"Profundidad {profundidad}: error con lo que vio = {dec(err_entrena[i], 3)} · con lo que no vio = {dec(err_prueba[i], 3)}")
    if err_entrena[i] < 0.6 * err_prueba[i]:
        print("⚠️ El árbol se sabe el pasado de memoria, pero falla mucho más con datos nuevos: sobreajuste.")

jugar(sobreajuste, profundidad=deslizador("Profundidad", 1, 15, 1, 3))

El error de prueba más bajo está alrededor de la profundidad **5**. Desde ahí, el error de entrenamiento sigue bajando hasta llegar a **cero** (el árbol se aprendió cada trimestre de memoria), pero el error de prueba sube. Por eso nunca se juzga un modelo con los datos con los que aprendió.

---
## 6 · ¿Qué mira el modelo?

El bosque aleatorio lleva la cuenta de cuánto le sirvió cada pista para acertar: es la **importancia** de cada variable. Las importancias suman 1.

🎚️ **Juegue.** Cambie la variable que se pronostica. ¿Las pistas que más pesan tienen sentido económico?

In [ ]:
def importancia(variable="Desempleo"):
    objetivo = OBJETIVOS[variable]
    entrena, _ = preparar(objetivo, 2000)
    bosque = RandomForestRegressor(n_estimators=300, random_state=0).fit(entrena[PISTAS], entrena["siguiente"])
    imp = pd.Series(bosque.feature_importances_, index=PISTAS).sort_values()
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.barh(imp.index, imp, color=[CORAL if v == imp.max() else GRIS for v in imp])
    ax.set_xlabel("importancia (suman 1)")
    ax.set_title(f"Lo que más mira el bosque para pronosticar: {variable}")
    plt.show()
    print(f"La pista más importante es «{imp.idxmax()}», con {porcentaje(imp.max(), 1)} del total.")

jugar(importancia, variable=lista("Pronosticar", list(OBJETIVOS), "Desempleo"))

Para pronosticar el desempleo del próximo trimestre, la pista que más pesa es el **desempleo de hoy**: el mercado laboral cambia despacio. Una advertencia importante: la importancia dice qué le sirvió al modelo para **acertar**, no qué **causa** qué. Que una variable sea útil para pronosticar no quiere decir que, si se cambia, cambie el resultado.

---
## 7 · El torneo

Todos los modelos contra las tres variables, con corte en 2000. La celda tarda unos segundos: entrena quince modelos.

🔮 **Prediga** antes de ejecutar: ¿en qué variable gana el ingenuo?

In [ ]:
filas = {}
for variable, objetivo in OBJETIVOS.items():
    entrena, prueba = preparar(objetivo, 2000)
    columna = {"Ingenuo": mean_absolute_error(prueba["siguiente"], prueba["hoy"])}
    for nombre in MODELOS:
        m = crear_modelo(nombre).fit(entrena[PISTAS], entrena["siguiente"])
        columna[nombre] = mean_absolute_error(prueba["siguiente"], m.predict(prueba[PISTAS]))
    filas[variable] = columna

torneo = pd.DataFrame(filas).round(3)
display(torneo)
for variable in torneo.columns:
    print(f"{variable:>20}: gana {torneo[variable].idxmin()} (error medio {dec(torneo[variable].min(), 3)})")

**Tres lecciones del torneo.**

1. **El desempleo se deja pronosticar**: casi todos los modelos le ganan al ingenuo, porque sigue patrones (sube en las recesiones, baja cuando la economía crece).
2. **La inflación es difícil**: los modelos mejoran un poco al ingenuo, pero se siguen equivocando por casi dos puntos en promedio y su R² es negativo. Los precios dependen de cosas que no están en la tabla: el petróleo, las expectativas, decisiones del banco central.
3. **En el crecimiento del PIB gana el ingenuo**: con estas pistas, ningún modelo supera a «igual que hoy». Saber cuándo **no** se puede pronosticar también es un resultado.

---
## 8 · Simulador: invente una economía

Aquí usted es el analista. Mueva las condiciones de **hoy** y el modelo, entrenado con todo el periodo 1960-2009, le dice cuánto será el **desempleo del próximo trimestre**.

🎚️ **Juegue.** Pruebe una economía en auge (PIB creciendo 5 %, inversión 10 %) y una en crisis (PIB −3 %, inversión −20 %). ¿Cambia más el pronóstico con la regresión o con el bosque?

In [ ]:
SIM = ["desempleo", "crec_pib", "crec_inversion", "inflacion", "tasa_interes"]
datos_sim = eco[SIM].copy()
datos_sim["siguiente"] = eco["desempleo"].shift(-1)
datos_sim = datos_sim.dropna()
modelos_sim = {"Regresión lineal": LinearRegression().fit(datos_sim[SIM], datos_sim["siguiente"]),
               "Bosque aleatorio": RandomForestRegressor(n_estimators=300, random_state=0).fit(datos_sim[SIM], datos_sim["siguiente"])}

def simulador(modelo="Regresión lineal", desempleo_hoy=5.5, crec_pib=2.5, crec_inversion=3.0, inflacion=3.0, tasa_interes=4.0):
    escenario = pd.DataFrame([[desempleo_hoy, crec_pib, crec_inversion, inflacion, tasa_interes]], columns=SIM)
    p = modelos_sim[modelo].predict(escenario)[0]
    cambio = p - desempleo_hoy
    flecha = "⬆️ sube" if cambio > 0.05 else "⬇️ baja" if cambio < -0.05 else "➡️ casi igual"
    fig, ax = plt.subplots(figsize=(6, 2.6))
    ax.barh(["próximo trimestre", "hoy"], [p, desempleo_hoy], color=[CORAL, GRIS])
    ax.set_xlim(0, 12)
    ax.set_xlabel("desempleo, %")
    plt.show()
    print(f"{modelo}: el desempleo pasaría de {dec(desempleo_hoy, 1)} % a {dec(p, 2)} % ({flecha}, {dec(cambio, 2)} puntos).")

jugar(simulador,
      modelo=lista("Modelo", list(modelos_sim)),
      desempleo_hoy=deslizador("Desempleo hoy (%)", 3.0, 11.0, 0.1, 5.5),
      crec_pib=deslizador("Crecimiento del PIB (%)", -4.0, 8.0, 0.5, 2.5),
      crec_inversion=deslizador("Crecimiento de la inversión (%)", -25.0, 20.0, 1.0, 3.0),
      inflacion=deslizador("Inflación (%)", -2.0, 12.0, 0.5, 3.0),
      tasa_interes=deslizador("Tasa de interés (%)", 0.0, 15.0, 0.25, 4.0))

🧭 **Reto.** Ponga un escenario que **nunca ocurrió** en los datos: desempleo de 3 % con el PIB cayendo 4 %. ¿Qué responde cada modelo? La regresión sigue su recta hasta donde haga falta; el bosque solo puede promediar trimestres que ya vio, así que nunca pronostica algo que no haya pasado. Ninguno de los dos sabe que ese escenario es raro: eso lo tiene que saber usted.

---
## 9 · Lo que un modelo no ve

Hoy vio modelos que le ganan al ingenuo con el desempleo y que pierden con el PIB. Vio a la curva de Phillips desaparecer en los setenta y a un árbol sabérselo todo de memoria sin entender nada. Tres responsabilidades para quien pronostica con *machine learning*:

- **Medir siempre contra un rival sencillo.** Un modelo complejo que no le gana al ingenuo no aporta nada, por sofisticado que suene.
- **Probar con datos que el modelo no vio**, y decir con cuáles se probó. Un error medido con los datos de entrenamiento es un error falso.
- **Recordar que el modelo aprende del pasado.** La crisis de 2008 no se parecía a nada de lo que había antes, y la pandemia de 2020 tampoco. Un pronóstico es una herramienta para decidir, no una profecía.

En 1976 el economista Robert Lucas advirtió algo más: cuando el gobierno cambia sus políticas, la gente cambia su comportamiento, y las relaciones que el modelo aprendió pueden dejar de valer. Los datos describen un mundo que ya cambió.

---
## Para seguir jugando

- En la parte 4, agregue la columna `"tasa_real"` a la lista `PISTAS` y vuelva a ejecutar desde ahí. ¿Mejora algún modelo?
- En `crear_modelo`, cambie `n_neighbors=5` por `n_neighbors=15`, o `max_depth=4` por `max_depth=2`. ¿Qué pasa en el torneo?
- Escriba en una celda nueva `sm.datasets.` y presione **Tab**: verá otras bases de datos que trae `statsmodels`. ¿Encuentra alguna de economía?

> Fuente de los datos: `statsmodels.datasets.macrodata`, compilada por Skipper Seabold con datos de FRED (Reserva Federal de San Luis) y de la Oficina de Estadísticas Laborales de Estados Unidos, consultados el 15 de diciembre de 2009. Recesiones: fechas del Comité de Fechado de Ciclos Económicos del NBER.